# NHS England Maternal Care Pathways Master Pipeline
## Stage 1 - Load MSDS demographics/booking data and aggregate to one entry per pregnancy

### Compiled by Ethan Phillips (Oxford)

### Last updated 10.11.2025


In [0]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR

In [0]:
df_demographic = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_demographics_booking_and_pregnancy_all_years"
)

In [0]:
df_demographic = df_demographic.withColumns(
    {
        'Rank_IMD_Decile_2015': when(col('Rank_IMD_Decile_2015')=='01 - Most deprived', 1).when(col('Rank_IMD_Decile_2015')=='10 - Least deprived', 10).otherwise(col('Rank_IMD_Decile_2015')).cast('int')
    }
)

In [0]:
df_demographic = df_demographic.withColumns({
    "ageatbookingmother": col("ageatbookingmother").try_cast('int'),
    "ageatlabourmother": col("ageatlabourmother").try_cast('int'),
    "ageatdeathmother": col("ageatdeathmother").try_cast('int'),

    "gestagebooking": col("gestagebooking").try_cast('int'),
    "gestagedatultradate": col("gestagedatultradate").try_cast('int'),

    "alcoholunitsbooking": col("alcoholunitsbooking").try_cast('float'),
    "personbmibooking": col("personbmibooking").try_cast('float'),

    "carecontact_attended_count": col("carecontact_attended_count").try_cast('int'),
    "carecontact_email_count": col("carecontact_email_count").try_cast('int'),
    "carecontact_facetoface_count": col("carecontact_facetoface_count").try_cast('int'),
    "carecontact_notattended_count": col("carecontact_notattended_count").try_cast('int'),
    "carecontact_other_count": col("carecontact_other_count").try_cast('int'),
    "carecontact_telephone_count": col("carecontact_telephone_count").try_cast('int'),
    "carecontact_sms_count": col("carecontact_sms_count").try_cast('int'),
    "carecontact_telemedicine_count": col("carecontact_telemedicine_count").try_cast('int'),
    "carecontact_talktype_count": col("carecontact_talktype_count").try_cast('int'),

    "previouscaesareansections": col("previouscaesareansections").try_cast('int'),
    "previouslivebirths": col("previouslivebirths").try_cast('int'),
    "previouslosseslessthan24weeks": col("previouslosseslessthan24weeks").try_cast('int'),
    "previousstillbirths": col("previousstillbirths").try_cast('int'),

})

In [0]:
features_to_keep = ["UniqPregID", "fyear"]

In [0]:
df_demographics_collapsed = df_demographic.groupBy('UniqPregID').agg(
    first('AdmMethCodeMothDelHospProvSpell', ignorenulls=True).alias('adm_meth_code_moth_del_hosp_prov_spell'),
    first('PostcodeDistrictMother', ignorenulls=True).alias('PostcodeDistrictMother'),
    first('LSOAMother2011', ignorenulls=True).alias('LSOAMother2011'),
    first('LAD_UAMother', ignorenulls=True).alias('LAD_UAMother'),
    first('person_id_mother_deid', ignorenulls=True).alias('Person_ID_DEID'),
    last('pseudo_uniquelabourdeliveryid', ignorenulls=True).alias('ld_id'),
    last('pseudo_uniquelocalfetalid', ignorenulls=True).alias('fetus_id'),
    max('fyear').alias('fyear'),

    first('ageatbookingmother', ignorenulls=True).alias('age_at_booking'),
    first('gestagebooking', ignorenulls=True).alias('gest_at_booking'),
    first('gestagedatultradate', ignorenulls=True).alias('gest_at_ultrasound'),
    first('ageatlabourmother', ignorenulls=True).alias('age_at_labour'),

    first('antepathlevel', ignorenulls=True).alias('ante_path_level'),
    first('orgidantpathicb', ignorenulls=True).alias('ante_path_icb'),
    first('orgidantpathsubicbloc', ignorenulls=True).alias('ante_path_loc'),
    last('postpathlevel', ignorenulls=True).alias('post_path_level'),
    last('orgidpostpathicb', ignorenulls=True).alias('post_path_icb'),
    last('orgidpostpathsubicbloc', ignorenulls=True).alias('post_path_loc'),

    first('offerstatusdatingultrasound', ignorenulls=True).alias('ultrasound_status'),
    first('leadanteprovider', ignorenulls=True).alias('antenatal_providerorg'),
    last('leadpostprovider', ignorenulls=True).alias('postnatal_providerorg'),

    last('ageatdeathmother').alias('age_at_death'),
    last('dayofweekofdeathmother', ignorenulls=True).cast('int').alias('death_day_of_week'),
    last('meridianofdeathmother', ignorenulls=True).cast('int').alias('death_am_or_pm'),
    last('persondeathtimemother', ignorenulls=True).alias('death_time'),
    last('monthofdeathmother', ignorenulls=True).cast('int').alias('death_month'),
    last('yearofdeathmother', ignorenulls=True).cast('int').alias('death_year'),

    max('alcoholunitsbooking').alias('alcohol_units'),
    first('alcoholunitsperweekband').alias('alcohol_units_band'),
    first('cigarettesperdayband').alias('cigarettes_band'),
    collect_set('folicacidsupplement').alias('folic_acid'),
    first('personbmiband').alias('bmi_band'),
    max('personbmibooking').alias('bmi'),

    first(to_date('pregfirstcondate', "yyyy-MM-dd"), ignorenulls=True).alias('first_contact_date'),
    first('pregfirstcontactcareproftype', ignorenulls=True).alias('first_contact_professional'),
    first('sourcerefmat', ignorenulls=True).alias('referral_source'),
    first(to_date('lastmenstrualperioddate', "yyy-MM-dd"), ignorenulls=True).alias('last_period_date'),
    first(to_date('antenatalappdate', "yyyy-MM-dd"), ignorenulls=True).alias('antenatal_appt_date'),
    first('orgsiteidbooking', ignorenulls=True).alias('booking_org'),
    first(to_date('activityofferdateultrasound', "yyyy-MM-dd"), ignorenulls=True).alias('ultrasound_offer_date'),
    first(to_date('proceduredatedatingultrasound', "yyyy-MM-dd"), ignorenulls=True).alias('ultrasound_date'),
    last(to_date('extractperiodenddate', "yyyy-MM-dd"), ignorenulls=True).alias('extract_end_date'),

    sum('carecontact_attended_count').alias('contacts_attended'),
    sum('carecontact_email_count').alias('contacts_email'),
    sum('carecontact_facetoface_count').alias('contacts_facetoface'),
    sum('carecontact_notattended_count').alias('contacts_not_attended'),
    sum('carecontact_other_count').alias('contacts_other'),
    sum('carecontact_telephone_count').alias('contacts_phone'),
    sum('carecontact_sms_count').alias('contacts_text'),
    sum('carecontact_telemedicine_count').alias('contacts_telemedicine'),
    sum('carecontact_talktype_count').alias('contacts_talktype'),

    last('ccgresidencemother', ignorenulls=True).alias('ccg_residence'),
    last('ccgresponsibilitymother', ignorenulls=True).alias('ccg_responsible'),
    last('countymother', ignorenulls=True).alias('county'),
    min('rank_imd_decile_2015').alias('imd_decile'),
    last(to_date('startdategmpreg', "yyyy-MM-dd"), ignorenulls=True).alias('start_gp_date'),
    first('orgcodegmpmother', ignorenulls=True).alias('gp_org_first'),
    last('orgcodegmpmother', ignorenulls=True).alias('gp_org_last'),
    first('orgidicbgppractice', ignorenulls=True).alias('gp_icb_first'),
    last('orgidicbgppractice', ignorenulls=True).alias('gp_icb_last'),
    first(to_date('enddategmpreg', "yyyy-MM-dd"), ignorenulls=True).alias('end_gp_date'),

    last('employmentstatusmother', ignorenulls=True).try_cast('int').alias('mother_employ_last'),
    first('employmentstatusmother', ignorenulls=True).try_cast('int').alias('mother_employ_first'),
    last('employmentstatuspartner', ignorenulls=True).try_cast('int').alias('partner_employ_last'),
    first('employmentstatuspartner', ignorenulls=True).try_cast('int').alias('partner_employ_first'),

    first('nofetusesdatingultrasound', ignorenulls=True).try_cast('int').alias('num_fetus'),
    first(to_date('eddagreed', "yyyy-MM-dd"), ignorenulls=True).alias('est_delivery_date'),
    first('eddmethodagreed', ignorenulls=True).alias('est_delivery_method'),
    last(to_date('decisiontodeliverdate', "yyyy-MM-dd"), ignorenulls=True).alias('delivery_decision_date'),
    last('decisiontodelivertime', ignorenulls=True).alias('delivery_decision_time'),
    last('birthsperlabanddel', ignorenulls=True).try_cast('int').alias('num_births'),
    last(to_date('caesareandate', "yyyy-MM-dd"), ignorenulls=True).alias('csection_date'),
    last('caesareantime', ignorenulls=True).alias('csection_time'),

    last('settingintracare', ignorenulls=True).alias('intrapartum_location'),

    first(to_date('startdatemotherdeliveryhospprovspell', "yyyy-MM-dd"), ignorenulls=True).alias('ld_hosp_start_date'),
    first('starttimemotherdeliveryhospprovspell', ignorenulls=True).alias('ld_hosp_start_time'),
    last(to_date('labouronsetdate', "yyyy-MM-dd"), ignorenulls=True).alias('labour_onset_date'),
    last('labouronsettime', ignorenulls=True).alias('labour_onset_time'),
    last('labouronsetmethod', ignorenulls=True).alias('labour_onset_method'),
    last('robsongroup', ignorenulls=True).try_cast('int').alias('robson'),
    last(to_date('dischargedatemotherhosp', "yyyy-MM-dd"), ignorenulls=True).alias('ld_disch_date'),
    last('dischargetimemotherhosp', ignorenulls=True).alias('ld_disch_time'),
    last(to_date('dischargedatematservice', "yyyy-MM-dd"), ignorenulls=True).alias('matserv_disch_date'),

    last('dischdestcodemothpostdelhospprovspell', ignorenulls=True).alias('ld_disch_dest'),
    last('dischmethcodemothpostdelhospprovspell', ignorenulls=True).alias('ld_disch_method'),
    last('dischreason', ignorenulls=True).alias('ld_disch_reason'),

    max('previouscaesareansections').alias('num_prev_csections'),
    max('previouslivebirths').alias('num_prev_births'),
    max('previouslosseslessthan24weeks').alias('num_prev_24w_losses'),
    max('previousstillbirths').alias('num_prev_still'),

    collect_set('abnormalitydatingultrasound').alias('abnormal_ultrasound'),
    collect_set('complexsocialfactorsind').alias('social_factors'),
    collect_set('disabilityindmother').alias('disability'),
    collect_set('ethniccategorymother').alias('ethnicity'),
    collect_set('langcode').alias('language'),
    collect_set('orgcodeprovider').alias('submitting_org'),
    collect_set('othersocpercircumstance').alias('social_personal_circumstance'),
    collect_set('religionsocpercircumstance').alias('religion'),
    collect_set('sexoriensocpercircumstance').alias('sexual_orientation'),
    collect_set('reasonchangedelsettinglab').alias('reason_ld_change'),
    collect_set('reasonlatebooking').alias('reason_late_booking'),
    collect_set('supportstatusindmother').alias('support_status')
)

# Add all aggregated column names to the master feature list
features_to_keep.extend([
    "adm_meth_code_moth_del_hosp_prov_spell","PostcodeDistrictMother", "LSOAMother2011", "LAD_UAMother", "Person_ID_DEID", "ld_id", "fetus_id",
    "age_at_booking", "gest_at_booking", "gest_at_ultrasound", "age_at_labour",
    "ante_path_level", "ante_path_icb", "ante_path_loc",
    "post_path_level", "post_path_icb", "post_path_loc",
    "ultrasound_status", "antenatal_providerorg", "postnatal_providerorg",
    "age_at_death", "death_day_of_week", "death_am_or_pm", "death_time", "death_month", "death_year",
    "alcohol_units", "alcohol_units_band", "cigarettes_band", "folic_acid", "bmi_band", "bmi",
    "first_contact_date", "first_contact_professional", "referral_source",
    "last_period_date", "antenatal_appt_date", "booking_org",
    "ultrasound_offer_date", "ultrasound_date",
    "ld_disch_date", "ld_disch_time", "matserv_disch_date", "extract_end_date",
    "contacts_attended", "contacts_email", "contacts_facetoface", "contacts_not_attended",
    "contacts_other", "contacts_phone", "contacts_text", "contacts_telemedicine", "contacts_talktype",
    "ccg_residence", "ccg_responsible", "county", "imd_decile",
    "start_gp_date", "gp_org_first", "gp_org_last", "gp_icb_first", "gp_icb_last", "end_gp_date",
    "mother_employ_last", "mother_employ_first", "partner_employ_last", "partner_employ_first",
    "num_fetus", "est_delivery_date", "est_delivery_method",
    "delivery_decision_date", "delivery_decision_time", "num_births",
    "csection_date", "csection_time", "intrapartum_location",
    "ld_hosp_start_date", "ld_hosp_start_time",
    "labour_onset_date", "labour_onset_time", "labour_onset_method",
    "robson", "ld_disch_dest", "ld_disch_method", "ld_disch_reason",
    "num_prev_csections", "num_prev_births", "num_prev_24w_losses", "num_prev_still",
    "abnormal_ultrasound", "social_factors", "disability", "ethnicity", "language",
    "submitting_org", "social_personal_circumstance", "religion", "sexual_orientation",
    "reason_ld_change", "reason_late_booking", "support_status"
])

In [0]:
df_demographics_collapsed = df_demographics_collapsed.select(features_to_keep)

In [0]:
print(
    f"Number of rows: {'{:,}'.format(df_demographics_collapsed.count())}. Number of columns: {'{:,}'.format(len(df_demographics_collapsed.columns))}"
)

In [0]:
write_table_name = "msds_demographics_agg_by_uniqpregid_stage"

df_demographics_collapsed.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")